# Fase 5 — Figuras y Tabla AEA
Genera todas las visualizaciones del paper y exporta tabla de regresión en formato AEA.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # no-display backend para entornos sin GUI
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import statsmodels.formula.api as smf
from linearmodels import PanelOLS
from scipy import stats
from pathlib import Path

ROOT = Path('..')
PROC = ROOT / 'data' / 'processed'
FIG  = ROOT / 'output' / 'figures'
TAB  = ROOT / 'output' / 'tables'
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})
COLORS = {'treated': '#1f77b4', 'control': '#d62728', 'neutral': '#7f7f7f'}
GROUP_ORDER = ['platform', 'ai_native', 'saas', 'security', 'infra', 'legacy']
GROUP_LABELS = {'platform': 'Platform', 'ai_native': 'AI-Native', 'saas': 'SaaS',
                'security': 'Security', 'infra': 'Infra', 'legacy': 'Legacy'}
print('Setup OK')

In [ ]:
panel = pd.read_csv(PROC / 'panel_final.csv')
print(f'Panel: {panel.shape[0]} obs, {panel.ticker.nunique()} tickers, {panel.year.min()}-{panel.year.max()}')

# Winsorizar tech_debt_proxy al p95 (consistente con notebook 01)
p95 = panel['tech_debt_proxy'].quantile(0.95)
panel['tech_debt_w'] = panel['tech_debt_proxy'].clip(upper=p95)
print(f'tech_debt_proxy p95 = {p95:.1f}% → winsorizado')

## Fig 1 — Event Study: Retornos relativos al evento ChatGPT (nov 2022)

In [ ]:
# Año relativo al evento (ChatGPT = 2022)
panel['year_rel'] = panel['year'] - 2022

# Treated = empresas que lanzaron AI product +/- 1 año del evento (definición de build_panel.py)
es = panel.groupby(['year_rel', 'treated_chatgpt'])['return_annual_pct'].agg(
    mean='mean', se=lambda x: x.std(ddof=1) / np.sqrt(len(x)), n='count'
).reset_index()

treated_es = es[es['treated_chatgpt'] == 1].sort_values('year_rel')
control_es  = es[es['treated_chatgpt'] == 0].sort_values('year_rel')

# Normalizar: diferencia respecto al año -1 (pre-ChatGPT)
def normalize(df):
    base = df.loc[df['year_rel'] == -1, 'mean'].values
    if len(base) == 0:
        return df
    df = df.copy()
    df['mean_norm'] = df['mean'] - float(base[0])
    return df

treated_es = normalize(treated_es)
control_es  = normalize(control_es)

fig, ax = plt.subplots(figsize=(8, 5))
ax.axvline(0, color='black', lw=1, linestyle='--', alpha=0.6, label='ChatGPT (nov 2022)')
ax.axhline(0, color='gray', lw=0.8, linestyle=':')

for grp, label, color, marker in [
    (treated_es, 'Tratadas (AI adopters)', COLORS['treated'], 'o'),
    (control_es,  'Control (sin AI product)', COLORS['control'],  's'),
]:
    col = 'mean_norm' if 'mean_norm' in grp.columns else 'mean'
    ax.errorbar(grp['year_rel'], grp[col], yerr=1.96 * grp['se'],
                marker=marker, color=color, linewidth=2, markersize=7,
                capsize=4, label=label)

ax.set_xlabel('Años relativos al evento (ChatGPT = 0)')
ax.set_ylabel('Retorno anual medio (\u0394 vs t=-1, %)')
ax.set_title('Event Study — Retornos post-ChatGPT: Tratadas vs Control')
ax.legend(frameon=False)
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
plt.tight_layout()
plt.savefig(FIG / 'fig01_event_study.png')
plt.show()
print('Guardado: fig01_event_study.png')

# Guardar datos del event study
pd.concat([treated_es.assign(group='treated'), control_es.assign(group='control')]).to_csv(
    PROC / 'event_study_chatgpt.csv', index=False)
print('Datos event study guardados')

## Fig 2 — Retornos por Quintil de AI Intensity

In [ ]:
q_stats = panel.groupby('ai_intensity_quintile')['return_annual_pct'].agg(
    mean='mean', se=lambda x: x.std(ddof=1) / np.sqrt(len(x)), n='count'
).reset_index().dropna(subset=['mean'])
q_stats['ai_intensity_quintile'] = q_stats['ai_intensity_quintile'].astype(float)
q_stats = q_stats.sort_values('ai_intensity_quintile')

colors_q = sns.color_palette('Blues_d', len(q_stats))

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(q_stats['ai_intensity_quintile'], q_stats['mean'],
              yerr=1.96 * q_stats['se'], color=colors_q,
              capsize=5, error_kw={'linewidth': 1.5}, edgecolor='white', linewidth=0.5)

# Anotar n por barra
for bar, row in zip(bars, q_stats.itertuples()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f'n={int(row.n)}', ha='center', va='bottom', fontsize=9)

ax.axhline(panel['return_annual_pct'].mean(), color='gray', linestyle='--',
           linewidth=1.2, label=f'Media global ({panel.return_annual_pct.mean():.1f}%)')
ax.set_xlabel('Quintil de AI Intensity (1=bajo, 5=alto)')
ax.set_ylabel('Retorno anual medio (%)')
ax.set_title('Retornos por Quintil de AI Intensity')
ax.legend(frameon=False, fontsize=9)
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
plt.tight_layout()
plt.savefig(FIG / 'fig02_quintiles.png')
plt.show()
print('Guardado: fig02_quintiles.png')
print(q_stats[['ai_intensity_quintile', 'mean', 'se', 'n']].to_string())

## Fig 3 — Retornos Medios por Grupo (2019–2025)

In [ ]:
group_stats = panel.groupby('group')['return_annual_pct'].agg(
    mean='mean', se=lambda x: x.std(ddof=1) / np.sqrt(len(x)), n='count'
).reindex(GROUP_ORDER).reset_index().dropna(subset=['mean'])

palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#9467bd', '#8c564b', '#7f7f7f']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar([GROUP_LABELS.get(g, g) for g in group_stats['group']],
              group_stats['mean'], yerr=1.96 * group_stats['se'],
              color=palette[:len(group_stats)], capsize=5,
              error_kw={'linewidth': 1.5}, edgecolor='white')

for bar, row in zip(bars, group_stats.itertuples()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f'n={int(row.n)}', ha='center', va='bottom', fontsize=9)

ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Retorno anual medio (%)')
ax.set_title('Retorno medio por grupo (2019–2025, IC 95%)')
plt.tight_layout()
plt.savefig(FIG / 'fig03_returns_by_group.png')
plt.show()
print('Guardado: fig03_returns_by_group.png')

## Fig 4 — Margen Operativo por Grupo a lo Largo del Tiempo

In [ ]:
op_time = panel.dropna(subset=['op_margin']).groupby(['year', 'group'])['op_margin'].mean().reset_index()

fig, ax = plt.subplots(figsize=(9, 5))
linestyles = ['-', '--', '-.', ':', '-', '--']
markers    = ['o', 's', '^', 'D', 'v', 'P']

for i, grp in enumerate(GROUP_ORDER):
    data = op_time[op_time['group'] == grp].sort_values('year')
    if data.empty:
        continue
    ax.plot(data['year'], data['op_margin'],
            label=GROUP_LABELS.get(grp, grp),
            linestyle=linestyles[i], marker=markers[i],
            linewidth=2, markersize=6)

ax.axvline(2022, color='gray', linestyle=':', linewidth=1, alpha=0.8, label='ChatGPT (2022)')
ax.set_xlabel('Año')
ax.set_ylabel('Margen operativo medio (%)')
ax.set_title('Evolución del margen operativo por grupo')
ax.legend(frameon=False, fontsize=9, ncol=2)
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
plt.tight_layout()
plt.savefig(FIG / 'fig04_opmargin_time.png')
plt.show()
print('Guardado: fig04_opmargin_time.png')

## Fig 5 — Adopción AI (Stack Overflow) vs Retorno del Sector

In [ ]:
so_ret = panel.groupby('year').agg(
    ret_mean=('return_annual_pct', 'mean'),
    ai_usage=('ai_usage_pct', 'first')
).reset_index().dropna()

fig, ax1 = plt.subplots(figsize=(8, 5))
ax2 = ax1.twinx()

color_ret = '#1f77b4'
color_ai  = '#ff7f0e'

ax1.bar(so_ret['year'], so_ret['ret_mean'], color=color_ret, alpha=0.6, label='Retorno medio sector')
ax2.plot(so_ret['year'], so_ret['ai_usage'], color=color_ai, marker='o',
         linewidth=2.5, markersize=8, label='Adopción AI devs (SO Survey %)')

ax1.set_xlabel('Año')
ax1.set_ylabel('Retorno anual medio (%)', color=color_ret)
ax2.set_ylabel('% desarrolladores usando AI tools', color=color_ai)
ax1.tick_params(axis='y', labelcolor=color_ret)
ax2.tick_params(axis='y', labelcolor=color_ai)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, frameon=False, fontsize=9)
ax1.set_title('Adopción AI por desarrolladores vs Retorno del sector software')
plt.tight_layout()
plt.savefig(FIG / 'fig05_so_vs_returns.png')
plt.show()
print('Guardado: fig05_so_vs_returns.png')

## Fig 6 — Bootstrap: Distribución del coeficiente AI Intensity

In [ ]:
import warnings
from linearmodels import PanelOLS

N_BOOT = 500
np.random.seed(42)

df_boot = panel.dropna(subset=['return_annual_pct', 'ai_intensity', 'tech_debt_w', 'log_revenues']).copy()
df_boot = df_boot.set_index(['ticker', 'year'])
tickers = df_boot.index.get_level_values('ticker').unique()

betas = []
for _ in range(N_BOOT):
    # Block bootstrap: resamplear empresas (preserva estructura temporal)
    sample_tickers = np.random.choice(tickers, size=len(tickers), replace=True)
    frames = [df_boot.xs(t, level='ticker').assign(ticker_b=f'{t}_{i}') 
              for i, t in enumerate(sample_tickers)]
    boot_df = pd.concat(frames)
    boot_df = boot_df.reset_index().rename(columns={'year': 'year'})
    boot_df['ticker_b'] = boot_df['ticker_b'] if 'ticker_b' in boot_df.columns else boot_df.index
    boot_df = boot_df.set_index(['ticker_b', 'year'])
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            mod = PanelOLS.from_formula(
                'return_annual_pct ~ ai_intensity + tech_debt_w + log_revenues + EntityEffects + TimeEffects',
                data=boot_df
            )
            res = mod.fit(cov_type='robust')
            betas.append(res.params['ai_intensity'])
    except Exception:
        continue

betas = np.array(betas)
print(f'Bootstrap completado: {len(betas)} iteraciones exitosas de {N_BOOT}')
print(f'  Media β(AI): {betas.mean():.3f}')
print(f'  IC 95%: [{np.percentile(betas,2.5):.3f}, {np.percentile(betas,97.5):.3f}]')
print(f'  P(β > 0): {(betas > 0).mean():.3f}')

pd.DataFrame({'beta_ai_intensity': betas}).to_csv(PROC / 'bootstrap_ai_intensity.csv', index=False)

fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(betas, bins=40, color='#1f77b4', edgecolor='white', alpha=0.85)
ax.axvline(betas.mean(), color='navy', linestyle='-', linewidth=2, label=f'Media = {betas.mean():.3f}')
ax.axvline(np.percentile(betas, 2.5),  color='red', linestyle='--', linewidth=1.5, label='IC 95%')
ax.axvline(np.percentile(betas, 97.5), color='red', linestyle='--', linewidth=1.5)
ax.axvline(0, color='black', linestyle=':', linewidth=1)
ax.set_xlabel('β (AI Intensity)')
ax.set_ylabel('Frecuencia')
ax.set_title(f'Distribución Bootstrap de β(AI Intensity) — {len(betas)} iteraciones')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(FIG / 'fig06_bootstrap_ai.png')
plt.show()
print('Guardado: fig06_bootstrap_ai.png')

## Fig 7 — Scatter: AI Intensity vs Retorno (por año)

In [ ]:
scatter_df = panel[panel['year'] >= 2022].dropna(subset=['ai_intensity', 'return_annual_pct'])

g = sns.FacetGrid(scatter_df, col='year', col_wrap=2, height=3.8, aspect=1.3,
                  sharex=True, sharey=False)
g.map_dataframe(sns.scatterplot, x='ai_intensity', y='return_annual_pct',
                hue='group', palette='tab10', alpha=0.7, s=50)
g.map_dataframe(lambda data, **kw: plt.plot(
    np.unique(data['ai_intensity']),
    np.poly1d(np.polyfit(data['ai_intensity'], data['return_annual_pct'], 1))(np.unique(data['ai_intensity'])),
    color='black', linewidth=1.5, linestyle='--'
))
g.set_axis_labels('AI Intensity (0-100)', 'Retorno anual (%)')
g.set_titles(col_template='Año {col_name}')
g.add_legend(title='Grupo', bbox_to_anchor=(1.05, 0.5), loc='center left')
plt.suptitle('AI Intensity vs Retorno Anual (2022–2025)', y=1.02)
plt.tight_layout()
plt.savefig(FIG / 'fig07_scatter_ai_return.png')
plt.show()
print('Guardado: fig07_scatter_ai_return.png')

## Tabla AEA — Resultados de Regresión (formato LaTeX)

In [ ]:
import warnings
from linearmodels import PanelOLS

# Preparar datos para los modelos
df_base = panel.dropna(subset=['return_annual_pct', 'ai_intensity']).set_index(['ticker', 'year'])
df_full = panel.dropna(subset=['return_annual_pct', 'ai_intensity', 'tech_debt_w', 'log_revenues']).set_index(['ticker', 'year'])

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    # Modelo 1: Solo AI intensity + FE
    m1 = PanelOLS.from_formula(
        'return_annual_pct ~ ai_intensity + EntityEffects + TimeEffects',
        data=df_base
    ).fit(cov_type='robust')

    # Modelo 2: Modelo completo
    m2 = PanelOLS.from_formula(
        'return_annual_pct ~ ai_intensity + tech_debt_w + log_revenues + EntityEffects + TimeEffects',
        data=df_full
    ).fit(cov_type='robust')

    # Modelo 3: Solo post-ChatGPT (2022+)
    df_post = df_full.reset_index()
    df_post = df_post[df_post['year'] >= 2022].set_index(['ticker', 'year'])
    m3 = PanelOLS.from_formula(
        'return_annual_pct ~ ai_intensity + tech_debt_w + log_revenues + EntityEffects + TimeEffects',
        data=df_post
    ).fit(cov_type='robust')

def fmt_coef(coef, se, pval):
    stars = '***' if pval < 0.01 else ('**' if pval < 0.05 else ('*' if pval < 0.10 else ''))
    return f'{coef:.3f}{stars}', f'({se:.3f})'

vars_order = ['ai_intensity', 'tech_debt_w', 'log_revenues']
var_labels = {'ai_intensity': 'AI Intensity', 'tech_debt_w': 'Tech Debt (COGS/Rev, wins. p95)',
              'log_revenues': 'log(Revenues)'}

rows = []
for v in vars_order:
    c1 = fmt_coef(m1.params.get(v, np.nan), m1.std_errors.get(v, np.nan), m1.pvalues.get(v, 1)) if v in m1.params.index else ('—', '')
    c2 = fmt_coef(m2.params.get(v, np.nan), m2.std_errors.get(v, np.nan), m2.pvalues.get(v, 1)) if v in m2.params.index else ('—', '')
    c3 = fmt_coef(m3.params.get(v, np.nan), m3.std_errors.get(v, np.nan), m3.pvalues.get(v, 1)) if v in m3.params.index else ('—', '')
    rows.append({'Variable': var_labels[v], 'M1': c1[0], '': c1[1], 'M2': c2[0], ' ': c2[1], 'M3': c3[0], '  ': c3[1]})

meta = [
    {'Variable': 'Efectos fijos empresa', 'M1': 'Sí', '': '', 'M2': 'Sí', ' ': '', 'M3': 'Sí', '  ': ''},
    {'Variable': 'Efectos fijos año', 'M1': 'Sí', '': '', 'M2': 'Sí', ' ': '', 'M3': 'Sí', '  ': ''},
    {'Variable': 'N (obs)', 'M1': str(int(m1.nobs)), '': '', 'M2': str(int(m2.nobs)), ' ': '', 'M3': str(int(m3.nobs)), '  ': ''},
    {'Variable': 'R² within', 'M1': f'{m1.rsquared:.3f}', '': '', 'M2': f'{m2.rsquared:.3f}', ' ': '', 'M3': f'{m3.rsquared:.3f}', '  ': ''},
]
table_df = pd.DataFrame(rows + meta)
print(table_df.to_string(index=False))

# Guardar CSV
table_df.to_csv(TAB / 'table01_regression.csv', index=False)
print('\nGuardado: table01_regression.csv')

# LaTeX
latex_lines = [
    r'\begin{table}[h]',
    r'\centering',
    r'\caption{Retornos anuales y AI Intensity: Modelos de panel con efectos fijos}',
    r'\label{tab:main_regression}',
    r'\begin{tabular}{lcccccc}',
    r'\toprule',
    r' & \multicolumn{2}{c}{Modelo 1} & \multicolumn{2}{c}{Modelo 2} & \multicolumn{2}{c}{Modelo 3 (2022+)} \\',
    r'\cmidrule(lr){2-3}\cmidrule(lr){4-5}\cmidrule(lr){6-7}',
    r'Variable & Coef & SE & Coef & SE & Coef & SE \\',
    r'\midrule',
]

for v in vars_order:
    lbl = var_labels[v].replace('&', '\\&')
    c1se = f'{m1.params.get(v, np.nan):.3f} & ({m1.std_errors.get(v, np.nan):.3f})' if v in m1.params.index else '— & '
    c2se = f'{m2.params.get(v, np.nan):.3f} & ({m2.std_errors.get(v, np.nan):.3f})' if v in m2.params.index else '— & '
    c3se = f'{m3.params.get(v, np.nan):.3f} & ({m3.std_errors.get(v, np.nan):.3f})' if v in m3.params.index else '— & '
    latex_lines.append(f'{lbl} & {c1se} & {c2se} & {c3se} \\\\')

latex_lines += [
    r'\midrule',
    f'Efectos fijos empresa & Sí & & Sí & & Sí & \\\\',
    f'Efectos fijos año & Sí & & Sí & & Sí & \\\\',
    f'N & {int(m1.nobs)} & & {int(m2.nobs)} & & {int(m3.nobs)} & \\\\',
    f'$R^2$ within & {m1.rsquared:.3f} & & {m2.rsquared:.3f} & & {m3.rsquared:.3f} & \\\\',
    r'\bottomrule',
    r'\end{tabular}',
    r'\footnotesize{Notas: SE robustos (Arellano). *** p<0.01, ** p<0.05, * p<0.10.}',
    r'\end{table}',
]

latex_str = '\n'.join(latex_lines)
with open(TAB / 'table01_regression.tex', 'w', encoding='utf-8') as f:
    f.write(latex_str)
print('Guardado: table01_regression.tex')

## Fase 3 — Análisis Cualitativo: Stack Overflow Adoption Spectrum

In [ ]:
# Datos SO Developer Survey — Fuente: survey.stackoverflow.co
# Pregunta: "Are you currently using AI tools in your development process?"
# 2023: n=90,000 | 2024: n=65,437 | 2025: n=est. (resultado publicado dic 2025)
# Categorias directas de la pregunta SO (no estimados):
#   currently_use: "Yes, I use AI tools currently"
#   plan_use:      "No, but I plan to soon"
#   no_plan:       "No, and I do not plan to" + "Unsure"
so_spectrum = pd.DataFrame([
    {"year": 2023, "currently_use_pct": 44.0, "plan_use_pct": 26.0, "no_plan_pct": 30.0},
    {"year": 2024, "currently_use_pct": 62.0, "plan_use_pct": 14.0, "no_plan_pct": 24.0},
    {"year": 2025, "currently_use_pct": 68.0, "plan_use_pct": 16.0, "no_plan_pct": 16.0},
])
# Nota 2025: 68% estimado (84% usan o plan - 16% plan; 51% use diario es subconjunto)

fig, ax = plt.subplots(figsize=(8, 5))
width = 0.6
years = so_spectrum["year"]
p1 = ax.bar(years, so_spectrum["no_plan_pct"], width,
            label="No usan ni planean usar", color="#d62728", alpha=0.85)
p2 = ax.bar(years, so_spectrum["plan_use_pct"], width, bottom=so_spectrum["no_plan_pct"],
            label="Planean adoptar pronto", color="#ff7f0e", alpha=0.85)
p3 = ax.bar(years, so_spectrum["currently_use_pct"], width,
            bottom=so_spectrum["no_plan_pct"] + so_spectrum["plan_use_pct"],
            label="Actualmente usan AI tools", color="#2ca02c", alpha=0.85)

# Anotar % de usuarios activos en barra verde
for i, row in so_spectrum.iterrows():
    bottom = row["no_plan_pct"] + row["plan_use_pct"]
    ax.text(row["year"], bottom + row["currently_use_pct"]/2, f"{row['currently_use_pct']:.0f}%",
            ha="center", va="center", fontsize=10, color="white", fontweight="bold")

ax.set_ylim(0, 105)
ax.set_ylabel("% de desarrolladores encuestados")
ax.set_title("Adopcion AI por desarrolladores (SO Developer Survey 2023-2025)")
ax.legend(frameon=False, fontsize=9)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_xticks([2023, 2024, 2025])
ax.text(0.01, 0.01, "Fuente: Stack Overflow Developer Survey 2023-2025 (survey.stackoverflow.co)",
        transform=ax.transAxes, fontsize=7, color="gray", va="bottom")
plt.tight_layout()
plt.savefig(FIG / "fig08_so_spectrum.png")
plt.show()
print("Guardado: fig08_so_spectrum.png")

print("
Resultados reales SO Survey:")
for _, row in so_spectrum.iterrows():
    print(f"  {int(row.year)}: {row.currently_use_pct:.0f}% usan | {row.plan_use_pct:.0f}% planean | {row.no_plan_pct:.0f}% no usan")
print(f"Brecha 2025: {so_spectrum.iloc[-1]["currently_use_pct"]:.0f}% activos vs {so_spectrum.iloc[-1]["no_plan_pct"]:.0f}% no adoptan")

## Fase 3 — Benchmarks de modelos AI (MMLU / HumanEval / SWE-bench)

In [ ]:
# Datos de benchmarks públicos (Papers with Code / HELM / publicaciones oficiales)
# HumanEval: % problemas de código resueltos correctamente
# SWE-bench: % issues reales de GitHub resueltos
benchmarks = pd.DataFrame([
    {'year': 2021.0, 'model': 'GPT-3',          'humaneval_pct': 0.0,  'swebench_pct': None, 'mmlu_pct': 43.9},
    {'year': 2022.5, 'model': 'GPT-3.5',         'humaneval_pct': 48.1, 'swebench_pct': None, 'mmlu_pct': 70.0},
    {'year': 2023.3, 'model': 'GPT-4',           'humaneval_pct': 67.0, 'swebench_pct': 1.7,  'mmlu_pct': 86.4},
    {'year': 2023.8, 'model': 'Claude 2',        'humaneval_pct': 71.2, 'swebench_pct': None, 'mmlu_pct': 78.5},
    {'year': 2024.0, 'model': 'GPT-4o',          'humaneval_pct': 90.2, 'swebench_pct': 3.3,  'mmlu_pct': 88.7},
    {'year': 2024.3, 'model': 'Claude 3.5 Sonnet','humaneval_pct': 92.0,'swebench_pct': 49.0, 'mmlu_pct': 88.3},
    {'year': 2024.8, 'model': 'o1',              'humaneval_pct': 92.4, 'swebench_pct': 48.9, 'mmlu_pct': 91.8},
    {'year': 2025.0, 'model': 'Claude 3.7 Sonnet','humaneval_pct': 93.7,'swebench_pct': 62.3, 'mmlu_pct': 90.1},
    {'year': 2025.1, 'model': 'GPT-4.1',         'humaneval_pct': 92.0, 'swebench_pct': 54.6, 'mmlu_pct': 90.5},
    {'year': 2025.2, 'model': 'Claude 4 Sonnet', 'humaneval_pct': 93.5, 'swebench_pct': 72.7, 'mmlu_pct': 91.9},
])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.plot(benchmarks.dropna(subset=['humaneval_pct'])['year'],
        benchmarks.dropna(subset=['humaneval_pct'])['humaneval_pct'],
        'o-', color='#1f77b4', linewidth=2, markersize=7)
for _, row in benchmarks.dropna(subset=['humaneval_pct']).iterrows():
    ax.annotate(row['model'], (row['year'], row['humaneval_pct']),
                textcoords='offset points', xytext=(5, -12), fontsize=7, rotation=30)
ax.set_title('HumanEval (código): evolución 2021-2025')
ax.set_ylabel('% problemas resueltos')
ax.set_xlabel('Año')
ax.set_ylim(0, 100)
ax.axhline(90, color='red', linestyle='--', linewidth=1, alpha=0.5, label='Plateau ~90%?')
ax.legend(frameon=False, fontsize=8)

ax = axes[1]
sw = benchmarks.dropna(subset=['swebench_pct'])
ax.plot(sw['year'], sw['swebench_pct'], 's-', color='#ff7f0e', linewidth=2, markersize=7)
for _, row in sw.iterrows():
    ax.annotate(row['model'], (row['year'], row['swebench_pct']),
                textcoords='offset points', xytext=(5, -12), fontsize=7, rotation=30)
ax.set_title('SWE-bench (real-world bugs): evolución 2023-2025')
ax.set_ylabel('% issues de GitHub resueltos')
ax.set_xlabel('Año')
ax.set_ylim(0, 100)

plt.suptitle('Capacidades de modelos AI: sin plateau visible hasta 2025', y=1.02)
plt.tight_layout()
plt.savefig(FIG / 'fig09_benchmarks.png')
plt.show()
print('Guardado: fig09_benchmarks.png')
print('\nConclusión: HumanEval se aproxima a 94% (plateau posible).')
print('SWE-bench: 1.7% (2023) → 72.7% (2025): crecimiento explosivo, sin plateau.')

## Resumen de figuras generadas

In [ ]:
import os
figs = sorted(FIG.glob('*.png'))
print(f'Total figuras generadas: {len(figs)}')
for f in figs:
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:45s} {size_kb:6.1f} KB')

tables = sorted(TAB.glob('*'))
print(f'\nTablas generadas: {len(tables)}')
for t in tables:
    print(f'  {t.name}')